# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [11]:
#Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, accuracy_score

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
#your code here
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [4]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
spaceship_cleaned = spaceship.dropna()
spaceship_cleaned['Cabin'] = spaceship_cleaned['Cabin'].str[0]
spaceship_cleaned = spaceship_cleaned.drop(columns=['Name', 'PassengerId'])
spaceship_encoded = pd.get_dummies(spaceship_cleaned, columns=['Cabin', 'HomePlanet', 'Destination', 'VIP', 'CryoSleep'], drop_first=True)

/var/folders/5l/d76xm4td69x828mtm3bkbxhh0000gn/T/ipykernel_4210/1698203710.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_cleaned['Cabin'] = spaceship_cleaned['Cabin'].str[0]


In [6]:
# Feature Scaling
features = spaceship_encoded.drop(columns=['Transported'])
target = spaceship_encoded['Transported']
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [7]:
#your code here
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled,
    target,
)

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

In [9]:
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)

GradientBoostingClassifier()

- Evaluate your model

In [12]:
#your code here
pred_gb = gb.predict(X_test)

print("Gradient Boosting accuracy:", accuracy_score(y_test, pred_gb), "\n")

print("Gradient Boosting RMSE:", root_mean_squared_error(y_test, pred_gb))
print("Gradient Boosting MAE:", mean_absolute_error(y_test, pred_gb))
print("Gradient Boosting R2 score:", r2_score(y_test, pred_gb))

Gradient Boosting accuracy: 0.8002421307506054 

Gradient Boosting RMSE: 0.446942803107282
Gradient Boosting MAE: 0.19975786924939468
Gradient Boosting R2 score: 0.2009579827252379


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
#your code here
# Lets use Grid Search to tune the hyperparameters of the Gradient Boosting model
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

- Run Grid Search

In [14]:
grid_search = GridSearchCV(estimator=gb, param_grid=param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)

Best parameters found:  {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}


- Evaluate your model

In [15]:
# Evaluate the best model from grid search
best_gb = grid_search.best_estimator_
pred_best_gb = best_gb.predict(X_test)

print("Best Gradient Boosting accuracy:", accuracy_score(y_test, pred_best_gb), "\n")

print("Best Gradient Boosting RMSE:", root_mean_squared_error(y_test, pred_best_gb))
print("Best Gradient Boosting MAE:", mean_absolute_error(y_test, pred_best_gb))
print("Best Gradient Boosting R2 score:", r2_score(y_test, pred_best_gb))

Best Gradient Boosting accuracy: 0.799636803874092 

Best Gradient Boosting RMSE: 0.44761947692868326
Best Gradient Boosting MAE: 0.200363196125908
Best Gradient Boosting R2 score: 0.19853664327895082


In [16]:
# Insights and conclusions
print("""Based on the results, we can see that hyperparameter tuning using Grid Search has decreased the accuracy of the Gradient Boosting model. 
The RMSE and MAE are similar before and after tuning, indicating that the average prediction error has not changed significantly. 
The R2 score has also slightly decreased, indicating a slight decrease in the proportion of variance explained by the model. 
Overall, while hyperparameter tuning is generally expected to improve model performance, in this case, it has led to a slight decrease in accuracy and R2 score.""")

Based on the results, we can see that hyperparameter tuning using Grid Search has decreased the accuracy of the Gradient Boosting model. 
The RMSE and MAE are similar before and after tuning, indicating that the average prediction error has not changed significantly. 
The R2 score has also slightly decreased, indicating a slight decrease in the proportion of variance explained by the model. 
Overall, while hyperparameter tuning is generally expected to improve model performance, in this case, it has led to a slight decrease in accuracy and R2 score.
